# Micrograd Parte 3 - Optimizadores

El objetivo de esta notebook es implementar los optimizadores **SGD** y **Adam**, y utilizarlos para entrenar nuestra red neuronal siguiendo el framework de Micrograd en el que venimos trabajando.

## Utils

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot


---
## Libreria de redes neuronales

Vamos a actualizar nuestra clase Value para agregar algunas operaciones más que serán útiles para más adelante:

### Operaciones adicionales
Se agregaron métodos para cubrir más operaciones.
- `__rsub__`: Permite que la resta funcione si el número esta a la izquiera (por ejemplo, `2 - x`)
- `__truediv__` y `__rtruediv__`: Permiten hacer divisiones (`/`) entre `Value` y `Value` o `Value` y números. También agregamos su backward.
- `log`: Agregamos la función `log` y su backward.

### Función Sigmoidea
Vamos a agregar tambien la función sigmoidea:
$$
\sigma(x) = \frac{1}{1+e^{-x}}
$$

Y su derivada:
$$
\sigma'(x) = \sigma(x) \cdot (1-\sigma(x))
$$



In [ ]:
# Nueva implementación de la clase Value
class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data}, label={self.label})"

  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')

    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward

    return out

  def __rmul__(self, other): # other * self
    return self * other

  def __neg__(self): # -self
    return self * -1

  def __sub__(self, other): # self - other
    return self + (-other)

  # Nuevo método
  def __rsub__(self, other): # other - self
    return other + (-self)

  def __radd__(self, other): # other + self
    return self + other

  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data**other, (self,), f'**{other}')

    def _backward():
        self.grad += other * (self.data ** (other - 1)) * out.grad
    out._backward = _backward

    return out

  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self, ), 'tanh')

    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward

    return out

  def sigmoid(self): # Implementamos la función sigmoidea
    x = self.data
    s = 1/(1 + math.exp(-x))
    out = Value(s, (self, ), 'sigmoid')

    def _backward():
      self.grad += (s * (1 - s)) * out.grad
    out._backward = _backward

    return out

  def log(self): # Implementamos la función logaritmo
    x = self.data
    out = Value(math.log(x), (self, ), 'log')

    def _backward():
      self.grad += (1/x) * out.grad
    out._backward = _backward

    return out

  # Nuevo método
  def __truediv__(self, other):  # self / other
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data / other.data, (self, other), '/')

    def _backward():
      self.grad += (1.0 / other.data) * out.grad
      other.grad += (-self.data / (other.data ** 2)) * out.grad
    out._backward = _backward

    return out

  # Nuevo método
  def __rtruediv__(self, other):  # other / self
    other = other if isinstance(other, Value) else Value(other)
    out = Value(other.data / self.data, (other, self), '/')

    def _backward():
      other.grad += (1.0 / self.data) * out.grad
      self.grad += (-other.data / (self.data ** 2)) * out.grad
    out._backward = _backward

    return out

  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

### Actualizamos la clase `Neuron` y `Layer`

Como ahora tenemos más de una función de activación, vamos a hacer que nuestra clase `Neuron` reciba como parámetro la función de activación que va a usar. La clase `Layer` también debe recibir la función de activación por parámetro ya que es quien se encarga de crear las neuronas de la red.

In [ ]:
import random

class Neuron:
  def __init__(self, nin, act=None):
    self.w = [Value(random.uniform(-1,1), label='w') for _ in range(nin)]
    self.b = Value(random.uniform(-1,1), label='b')

    if act is None:
      print("Warning! Neuron initialized without activation function")
    self.act = act

  def __call__(self, x): # Forward
    n = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)

    if self.act == "sigmoid":
      out = n.sigmoid()
    elif self.act == "tanh":
      out = n.tanh()
    else:
      out = n

    return out

  def parameters(self):
    return self.w + [self.b]

class Layer:
  def __init__(self, nin, nout, act=None): # La capa recibe una función de activación
    self.neurons = [Neuron(nin, act) for _ in range(nout)]

  def __call__(self, x):
    outs = [n(x) for n in self.neurons]
    return outs[0] if len(outs) == 1 else outs

  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]

In [ ]:
x = [2.0, 3.0]
n = Neuron(2, 'sigmoid')
out = n(x)
out

In [ ]:
draw_dot(out)

---
### Construimos nuestro MLP

<img src="https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fcs231n.github.io%2Fassets%2Fnn1%2Fneural_net2.jpeg&f=1&nofb=1&ipt=28a64b3c8e5d972ff118636e9e299aa4219291932c782694938ffad62c4c6dc3" width="500">



In [ ]:
class MLP:
  def __init__(self, input_size, nclasses, seed=None):
    if seed is not None:
      random.seed(seed)
    self.fc1 = Layer(input_size, 4, "tanh")
    self.fc2 = Layer(4, 4, "tanh")
    self.fc3 = Layer(4, nclasses, "tanh")

  def __call__(self, x):
    x = self.fc1(x)
    x = self.fc2(x)
    x = self.fc3(x)
    return x

  def parameters(self):
    fc1 = self.fc1.parameters()
    fc2 = self.fc2.parameters()
    fc3 = self.fc3.parameters()
    return fc1 + fc2 + fc3


## Optimizadores

Para el entrenamiento de una red neuronal usamos **algoritmos de optimización** que ajustan ajustan los parámetros siguiendo el gradiente. Existen muchas variantes de optimizadores, para este ejemplo vamos a utilizar **SGD** y **Adam**

### 1. SGD (Stochastic Gradient Descent)
Actualizar cada parámetros en la dirección contraria a su gradiente, escalado por una tasa de aprendizaje.

$$
\theta_{t+1} = \theta_t - \alpha\ \cdot\ g_t  
$$

- $\theta_t$: parámetros de la red en el paso $t$.
- $\alpha$: *learning_rate*
- $g_t$: gradiente respecto a la Loss

Cuando trabajamos con batchs (o mini-batchs), el gradiente se calcula solo con un lote de datos, lo que introduce ruido, de ahi el **Stochastic**.

**Variante con momentum**

Añadir una media móvil de gradientes:
$$
v_t = \beta v_{t-1} + (1-\beta) g_t \\
\theta_{t+1} = \theta_t - \alpha v_t
$$

In [ ]:
class SGD:
    def __init__(self, parameters, lr=0.01, momentum=0.0):
        self.parameters = parameters
        self.lr = lr
        self.momentum = momentum
        self.velocities = [0.0] * len(parameters) # Velocidades empiezan en cero

    def step(self):
        for i, p in enumerate(self.parameters):
            self.velocities[i] = self.momentum * self.velocities[i] + p.grad
            p.data -= self.lr * self.velocities[i]

    def zero_grad(self):
        for p in self.parameters:
            p.grad = 0.0

### 2. Adam (Adaptive Moment Estimation)
Mantiene promedios móviles de gradientes y sus cuadrados:
- $m_t$: promedio de gradientes (momento de primer orden)
- $v_t$: promedio de gradientes al cuadrado (momento de segundo orden)

Fórmulas:
1. Actualización de momentos:
$$
m_t = \beta_1 m_{t-1} + (1-\beta)g_t \\
v_t = \beta_2 v_{t-1} + (1-\beta)g_t^2
$$

1. Corrección por sesgo (importante en pasos iniciales):
$$
\hat{m}_t = \frac{m_t}{1-\beta_1^t},\ \ \hat{v}_t = \frac{v_t}{1-\beta_2^t},
$$

1. Actualización de parámetros:
$$
\theta_{t+1} = \theta_t - \alpha \cdot \frac{ \hat{m}_t}{\sqrt{\hat{v}_t} + ϵ}
$$

- $g_t$: gradiente en el paso $t$
- $\beta_1 ≈ 0.9, \beta_2 ≈ 0.999$
- $ϵ ≈ 1e^{-08}$: pequeño valor para evitar dividir por cero

In [ ]:
class Adam:
    def __init__(self, parameters, lr=0.001, betas=(0.9, 0.999), epsilon=1e-8):
        self.parameters = parameters
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.epsilon = epsilon
        self.m = [0.0] * len(parameters)
        self.v = [0.0] * len(parameters)
        self.t = 0

    def step(self):
        self.t += 1
        for i, p in enumerate(self.parameters):

            # Actualizar momentos
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * p.grad
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (p.grad ** 2)

            # Corrección por sesgo
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)

            # Actualizar parámetros
            p.data -= self.lr * m_hat / (math.sqrt(v_hat) + self.epsilon)

    def zero_grad(self):
        for p in self.parameters:
            p.grad = 0.0

## Dataset y funcion de Loss

In [ ]:
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

In [ ]:
def squared_loss(ys, ypreds):
  return sum((yout - ygt)**2 for ygt, yout in zip(ys, ypreds))

## Entrenamiento de la red

### Entrenamiento de la red con SGD

In [ ]:
mlp = MLP(3, 1, seed=42)
len(mlp.parameters())

In [ ]:
ypred = [mlp(x) for x in xs]

print("Predictions | Actual")
print("--------------------")
for pred, actual in zip(ypred, ys):
    print(f"{pred.data:.5f}     | {actual:.1f}")

In [ ]:
# Optimizador
learning_rate = 0.01
optimizer = SGD(mlp.parameters(), lr=learning_rate)

In [ ]:
epoch_loss = [] # colectamos el error para posterior análisis

In [ ]:
epochs = 50
for k in range(1, epochs+1):

  # forward pass
  ypred = [mlp(x) for x in xs]
  loss = squared_loss(ys, ypred)

  optimizer.zero_grad() # resetar gradientes

  # backward pass
  loss.backward()

  # update
  optimizer.step()

  epoch_loss.append(loss.data)

  # log every 10 epochs
  if k % 5 == 0:
    print(f"Epoch: {k:03d}/{epochs:03d} | Loss: {loss.data:.5f}")


In [ ]:
plt.plot(range(len(epoch_loss)), epoch_loss)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training Loss over Epochs")
plt.show()

In [ ]:
ypred = [mlp(x) for x in xs]

print("Predictions | Actual")
print("--------------------")
for pred, actual in zip(ypred, ys):
    print(f"{pred.data:.5f}     | {actual:.1f}")

In [ ]:
ypred = [mlp(x) for x in xs]
loss = squared_loss(ys, ypred)
loss

### Entrenamiento de la red con Adam

In [ ]:
mlp = MLP(3, 1, seed=42)
len(mlp.parameters())

In [ ]:
# Optimizador
learning_rate = 0.01
optimizer = Adam(mlp.parameters(), lr=learning_rate)

In [ ]:
epoch_loss = [] # colectamos el error para posterior análisis
epochs = 50
for k in range(1, epochs+1):

  # forward pass
  ypred = [mlp(x) for x in xs]
  loss = squared_loss(ys, ypred)

  optimizer.zero_grad() # resetar gradientes

  # backward pass
  loss.backward()

  # update
  optimizer.step()

  epoch_loss.append(loss.data)

  # log every 10 epochs
  if k % 10 == 0:
    print(f"Epoch: {k:03d}/{epochs:03d} | Loss: {loss.data:.5f}")


In [ ]:
plt.plot(range(len(epoch_loss)), epoch_loss)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training Loss over Epochs")
plt.show()

In [ ]:
ypred = [mlp(x) for x in xs]

print("Predictions | Actual")
print("--------------------")
for pred, actual in zip(ypred, ys):
    print(f"{pred.data:.5f}     | {actual:.1f}")

In [ ]:
ypred = [mlp(x) for x in xs]
loss = squared_loss(ys, ypred)
loss